# Berliner Straßennamen — Sprache, Geschlecht, Geschichte

Dieser Notebook analysiert die **Namen** der Berliner Straßen — unabhängig von Geometrie.
Kernthemen:
- Wie viele Straßen tragen Frauennamen? (nach Bezirk)
- Straßentypen und Namensstruktur
- Koloniale und NS-belastete Namen
- Kuriositäten: kürzeste, lustigste, längste Namen

**Datenquelle:** Straßenabschnitte RBS (strnr, strname, bezname)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import re

raw = gpd.read_file('../storytelling/data/strassen_berlin_wgs84.geojson')

# One row per unique street name × Bezirk
names = raw[['strname', 'strnr', 'bezname', 'ortname', 'strabtypname']].drop_duplicates(subset=['strname', 'bezname'])
# One row per unique street name (city-wide)
unique_names = raw[['strname']].drop_duplicates().reset_index(drop=True)

print(f'{len(unique_names):,} eindeutige Straßennamen')
print(f'{len(names):,} Straßenname × Bezirk Kombinationen')

## Frauen vs. Männer in Berliner Straßennamen

Nach Auswertungen (2025) liegt der Anteil von Straßen mit Frauennamen je nach Bezirk
nur bei wenigen Prozent: Mitte ~12,5 %, Pankow ~2,3 %.  
Wir reproduzieren diese Analyse aus den RBS-Straßennamen.

In [ ]:
# Common German female first names (top ~200) — extend as needed
FEMALE_NAMES = {
    'Anna', 'Marie', 'Maria', 'Elisabeth', 'Clara', 'Klara', 'Rosa', 'Emma', 'Luise', 'Louise',
    'Margarete', 'Margarethe', 'Käthe', 'Käte', 'Helene', 'Hedwig', 'Gertrude', 'Gertrud',
    'Else', 'Martha', 'Marta', 'Paula', 'Frieda', 'Friede', 'Hermine', 'Mathilde', 'Ottilie',
    'Wilhelmine', 'Auguste', 'Bertha', 'Berta', 'Charlotte', 'Christine', 'Eva', 'Johanna',
    'Hildegard', 'Irmgard', 'Ilse', 'Inge', 'Ingeborg', 'Irma', 'Josephine', 'Katharina',
    'Lena', 'Lina', 'Lisa', 'Lotte', 'Magdalena', 'Monika', 'Renate', 'Rita', 'Sophie',
    'Sofie', 'Susanne', 'Ursula', 'Veronika', 'Walburga', 'Wally', 'Wanda', 'Hilde',
    'Else', 'Elsa', 'Adelheid', 'Amalie', 'Antonie', 'Auguste', 'Babette', 'Dorothea',
    'Elfriede', 'Flora', 'Grete', 'Gretel', 'Hanna', 'Hannah', 'Ida', 'Julia', 'Juliane',
    'Karoline', 'Lisbeth', 'Leonore', 'Leopoldine', 'Mathilde', 'Minna', 'Nelly', 'Ottilie',
    'Pauline', 'Regine', 'Rosalie', 'Sidonie', 'Sibylle', 'Thea', 'Thekla', 'Theresia',
    'Trude', 'Trudi', 'Ulrike', 'Viktoria', 'Viola', 'Walpurga', 'Wilhelmine', 'Yvonne',
    # Historical / notable women
    'Rosa', 'Luxemburg', 'Käthe', 'Kollwitz', 'Clara', 'Zetkin', 'Bettina', 'Rahel',
    'Henriette', 'Hermine', 'Minna', 'Cauer', 'Helene', 'Lange', 'Gertrud', 'Alice',
    'Luise', 'Henriette', 'Amalie', 'Betty', 'Bella', 'Gisela', 'Marlene',
}

def likely_female(streetname):
    """Heuristic: does this street name start with a known female first name?"""
    # Remove type suffixes to get the name part
    cleaned = re.sub(
        r'(straße|strasse|-Str\.|allee|weg|platz|damm|chaussee|ring|gasse|steig|pfad|zeile|promenade)$',
        '', streetname, flags=re.IGNORECASE
    ).strip().rstrip('-')
    # Check if the first word (or hyphenated last part) is a female name
    parts = re.split(r'[-\s]', cleaned)
    return any(p in FEMALE_NAMES for p in parts)

unique_names['is_female'] = unique_names['strname'].apply(likely_female)

total = len(unique_names)
female_count = unique_names['is_female'].sum()
print(f'Straßen mit wahrscheinlichem Frauennamen: {female_count} von {total} ({female_count/total*100:.1f} %)')
unique_names[unique_names['is_female']].head(20)

In [ ]:
# By Bezirk
names['is_female'] = names['strname'].apply(likely_female)

gender_by_bez = names.groupby('bezname').agg(
    total=('strname', 'count'),
    weiblich=('is_female', 'sum')
)
gender_by_bez['anteil_pct'] = (gender_by_bez['weiblich'] / gender_by_bez['total'] * 100).round(1)
gender_by_bez = gender_by_bez.sort_values('anteil_pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
gender_by_bez['anteil_pct'].sort_values().plot(kind='barh', ax=ax, color='#e74c3c')
ax.axvline(gender_by_bez['anteil_pct'].mean(), color='gray', linestyle='--', label='Durchschnitt')
ax.set_title('Anteil von Straßen mit Frauennamen nach Bezirk (%)')
ax.set_xlabel('Anteil (%)')
ax.legend()
plt.tight_layout()
plt.show()

print(gender_by_bez)

## Koloniale Straßennamen

Bekannte Straßen mit kolonialen Bezügen — insbesondere im Afrikanischen Viertel (Wedding)
und Straßen, die nach Kolonialoffizieren oder -territorien benannt sind.

In [ ]:
COLONIAL_KEYWORDS = [
    'Lüderitz', 'Nachtigal', 'Petersallee', 'Togo', 'Kamerun', 'Kongo', 'Nama',
    'Wissmann', 'Lettow', 'Wörmann', 'Bismarck', 'Mohren', 'Anton-Wilhelm-Amo'
]

colonial = raw[raw['strname'].str.contains('|'.join(COLONIAL_KEYWORDS), case=False, na=False)]
colonial_names = colonial[['strname', 'bezname', 'ortname']].drop_duplicates(subset='strname')
print(f'{len(colonial_names)} Straßen mit kolonialen Schlagworten:')
colonial_names.sort_values('bezname')

## Straßen nach Namensstruktur

Berlin hat viele Straßen, die nach Städten, Ländern oder Regionen benannt sind — Relikte
einer Zeit, als Straßen Wegweiser in die Welt waren.

In [ ]:
# Streets named after German cities (ending in -er + street suffix)
city_streets = unique_names[unique_names['strname'].str.match(r'^[A-ZÄÖÜ][a-zäöü]+er (Straße|Allee|Weg|Damm|Platz|Chaussee)', na=False)]
print(f'{len(city_streets)} Straßen nach Städtemuster (z.B. "Potsdamer Str.", "Frankfurter Allee"):')
city_streets.head(30)

In [ ]:
# Streets starting with 'Alt-' (former village names)
alt_streets = unique_names[unique_names['strname'].str.startswith('Alt-', na=False)]
print(f'{len(alt_streets)} Straßen mit Alt-Präfix (ehemalige Dorfkerne):')
alt_streets

## Längste und kürzeste Straßennamen

In [ ]:
unique_names['nchar'] = unique_names['strname'].str.len()
real = unique_names[~unique_names['strname'].str.startswith(('BLW', 'Verbindungsweg'), na=False)]

print('=== Längste Straßennamen ===')
display(real.nlargest(10, 'nchar')[['strname', 'nchar']])

print('\n=== Kürzeste Straßennamen ===')
display(real.nsmallest(10, 'nchar')[['strname', 'nchar']])

## Kuriositäten: Weg A bis Weg G

In [ ]:
# Administrative placeholders that made it into the official registry
weg_letters = raw[raw['strname'].str.match(r'^Weg [A-Z]$', na=False)][['strname', 'bezname', 'ortname']].drop_duplicates()
print('Wege mit Buchstabenbezeichnung (Verwaltungslogik im Straßenregister):')
weg_letters.sort_values('strname')

In [ ]:
# The Kolk — shortest name, oldest alley in Spandau
kolk = raw[raw['strname'] == 'Kolk'][['strname', 'bezname', 'ortname', 'strabtypname']]
print('Der Kolk — eine der ältesten Gassen Berlins:')
kolk

## Straßentypen: Berlins preußisches Erbe

**Chaussee** und **Damm** sind in Berlin besonders häufig — typisch für norddeutsche und preußische Stadtplanung.
**Chaussee** bezeichnet eine gepflasterte Kunststraße (aus dem Französischen), **Damm** einen erhöhten Weg über sumpfiges Gelände.

In [ ]:
SUFFIXES = {
    'Straße/Str.': r'(straße|strasse|str\.)$',
    'Weg': r'weg$',
    'Allee': r'allee$',
    'Platz': r'(platz|pl\.)$',
    'Damm': r'damm$',
    'Chaussee': r'chaussee$',
    'Ring': r'ring$',
    'Gasse': r'gasse$',
    'Steig': r'steig$',
    'Promenade': r'promenade$',
}

counts = {}
for label, pattern in SUFFIXES.items():
    counts[label] = unique_names['strname'].str.contains(pattern, case=False, na=False).sum()
counts['Andere'] = len(unique_names) - sum(counts.values())

type_series = pd.Series(counts).sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
type_series.drop('Andere').sort_values().plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_title('Straßentypen in Berlin (ohne "Andere")')
ax1.set_xlabel('Anzahl eindeutiger Namen')

type_series.plot(kind='pie', ax=ax2, autopct='%1.1f%%', startangle=140,
                 colors=plt.cm.tab10.colors)
ax2.set_title('Verteilung (inkl. Andere)')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()
print(type_series)